## Building a Machine Learning Model to Predict cLogD
In this notebook we will use the descriptors we calculated using the **dask_descriptors.py** script build a model to predict cLogD.  First we'll import the necessary libraries. 

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, Booster
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import chembl_downloader
import pystow

Enable Pandas progress_apply

In [2]:
tqdm.pandas()

In [3]:
chembl_version = chembl_downloader.latest()
print(f"The latest version of ChEMBL is {chembl_version}")
chembl_version = "35"

The latest version of ChEMBL is 36


Note: ChEMBL 36 deprecated cx_clogd. `dask_descriptors.py` should have downloaded and processed an earlier version of chembl, which is set above.

In [4]:
from constants import SFI_MODULE
SFI_MODULE = SFI_MODULE.module(chembl_version)

### 1. Read the Descriptors
Read the data generated by **dask_descriptors.py**. 

In [5]:
%%time
df = pd.read_pickle(SFI_MODULE.join(name="logd_descriptors.pkl"))

CPU times: user 4.11 s, sys: 1.1 s, total: 5.21 s
Wall time: 5.95 s


Let's see how much data we read. 

In [6]:
df.shape

(2474590, 4)

As a sanity check, let's look at the first few rows.

In [7]:
df.head()

,smiles,name,logd,desc
0,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl,1,2.69,"[2.544219583800633, 2.151684657491087, -2.0923..."
1,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1,2,1.82,"[2.6098812317989752, 2.130900770307038, -2.085..."
2,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1,3,2.64,"[2.542112055121485, 2.170842647388084, -2.1841..."
3,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1,4,1.97,"[2.5513982230438503, 2.0932124840239217, -2.05..."
4,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1,5,2.57,"[2.546647604663282, 2.1295045180424377, -2.085..."


Drop any data with nulls. 

In [8]:
df.dropna(inplace=True)

Let's see how much data remains.  In practice, we'd try to figure out why some molecules didn't generate descriptors.  In this case, we still have more that 2 million records, so we're fine. 

In [9]:
df.shape

(2409279, 4)

Convert the descriptors from lists to numpy arrays. In retrospect, I may have been able to do this when I generated, the descriptors.  Oh well, not a big deal. 

### 2. Build the ML Model
Split the data into training and test sets. 

In [10]:
%%time
train, test = train_test_split(df)

CPU times: user 158 ms, sys: 11.7 ms, total: 170 ms
Wall time: 169 ms


In [14]:
len(train), len(test)

(1806959, 602320)

Use [np.stack](https://numpy.org/doc/stable/reference/generated/numpy.stack.html) to convert the descriptors to an appropriate format for ML model building. 

In [29]:
%%time
X_train = np.stack(train.desc.values)
y_train = train.logd.values
X_test = np.stack(test.desc.values)
y_test = test.logd.values

CPU times: user 4.8 s, sys: 5.95 s, total: 10.7 s
Wall time: 13.1 s


Build an ML model.  Wow, LightGBM is fast! 

In [16]:
%%time
lgbm = LGBMRegressor()
lgbm.fit(X_train,y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.180585 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27224
[LightGBM] [Info] Number of data points in the train set: 1806959, number of used features: 214
[LightGBM] [Info] Start training from score 2.481695
CPU times: user 1min 7s, sys: 7.01 s, total: 1min 14s
Wall time: 17.4 s


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


Predict on the test set.

In [17]:
%%time
pred = lgbm.predict(X_test)

/Users/vzhao/miniforge3/envs/sfi/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


CPU times: user 5.12 s, sys: 1.25 s, total: 6.37 s
Wall time: 2.75 s


### 3. Test the ML Model
Calculate $R^2$

In [18]:
r2_score(y_test,pred)

0.8979000872249658

Calculate RMSE

In [19]:
root_mean_squared_error(y_test,pred)

0.7945313115660113

### 3b. Retrain on all data

Generate a model trained on all data.

In [ ]:
# If low on memory
del X_train, y_train

In [23]:
X_all = np.stack(df.desc.values)
y_all = df.logd.values

In [25]:
%%time
lgbm = LGBMRegressor()
lgbm.fit(X_all,y_all)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.229517 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27240
[LightGBM] [Info] Number of data points in the train set: 2409279, number of used features: 217
[LightGBM] [Info] Start training from score 2.481103
CPU times: user 1min 19s, sys: 7.88 s, total: 1min 27s
Wall time: 20.6 s


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [34]:
pred = lgbm.predict(X_test)

/Users/vzhao/miniforge3/envs/sfi/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [35]:
r2_score(y_test,pred)

0.8991932533795848

In [36]:
root_mean_squared_error(y_test,pred)

0.789483632694462

### 4. Save the ML Model
Save the model to disk.

In [26]:
model_file = SFI_MODULE.join(name="model.txt")
lgbm.booster_.save_model(model_file)

Read the model from disk.

In [28]:
mdl = Booster(model_file=model_file)

Predict with the saved model and calculate $R^2$

In [30]:
pred = mdl.predict(X_test)

/Users/vzhao/miniforge3/envs/sfi/lib/python3.11/site-packages/lightgbm/basic.py:204: RuntimeWarning: overflow encountered in cast
  data = np.asarray(mat, dtype=dtype, order=order)


In [33]:
root_mean_squared_error(y_test,pred)

0.7894392102206491